In [1]:
%pip install pandas great-expectations pyyaml openpyxl

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/5.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.7 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.7 MB 932.9 kB/s eta 0:00:06
   ----- ---------------------------------- 0.8/5.7 MB 882.6 kB/s eta 0:00:06
   ------- -------------------------------- 1.0/5.7 MB 914.5 kB/s eta 0:00:06
   ------- -------------------------------- 1.0/5.7 MB 914.5 kB/s eta 0:00:06
   --------- ------------------------------ 1.3/5.7 MB 838.9 kB/s eta 0:00:06
   --------- ------------------------------ 1.3/5.7 MB 838.9 kB/s eta 0:00:06
   ---------- ----------------------------- 1.6/5.7 MB 755.5 kB/s eta 0:00:06
   ---------- ----------------------------- 1.6/5.7 MB 755.5 kB/s eta 0:00:06
   ------------ --------------------------- 1.8/5.7 MB 734.6 kB/s eta 0:00:06
   ----------

In [3]:
%pip install great-expectations

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import great_expectations as gx
import yaml

print("Pandas:", pd.__version__)
print("Great Expectations:", gx.__version__)
print("PyYAML:", yaml.__version__)

Pandas: 2.2.2
Great Expectations: 1.24.1
PyYAML: 6.0.1


In [2]:
from pathlib import Path
import pandas as pd

file_path = Path("cleaned_online_retail.csv")

if file_path.exists():
    df = pd.read_csv(file_path)
else:
    df = pd.read_excel("Online Retail.xlsx")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (536641, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nDuplicate rows:", df.duplicated().sum())

Column names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 536641 entries, 0 to 536640
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    536641 non-null  object 
 1   StockCode    536641 non-null  object 
 2   Description  535186 non-null  object 
 3   Quantity     536641 non-null  int64  
 4   InvoiceDate  536641 non-null  object 
 5   UnitPrice    536641 non-null  float64
 6   CustomerID   401604 non-null  float64
 7   Country      536641 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 32.8+ MB

Missing values:


,Missing Values
InvoiceNo,0
StockCode,0
Description,1455
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135037
Country,0



Duplicate rows: 0


In [4]:
required_columns = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are available.")
print("Dataset shape:", df.shape)
display(df.head())

All required columns are available.
Dataset shape: (536641, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
df = df.copy()

# Standardise column names
df.columns = df.columns.str.strip()

# Parse dates without silently hiding invalid values
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"], errors="coerce"
)

print("Working copy created.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Working copy created.
Rows: 536641
Columns: 8


In [6]:
missing_count = df.isna().sum()
missing_rate = (missing_count / len(df) * 100).round(2)

completeness_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Rate (%)": missing_rate
})

display(completeness_report)


,Missing Count,Missing Rate (%)
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1455,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135037,25.16
Country,0,0.00


In [7]:
import numpy as np

# Accuracy: proxy checks, not proof of real-world accuracy
accuracy_pass = (
    df["Quantity"].gt(0) &
    df["UnitPrice"].ge(0)
)
accuracy_score = accuracy_pass.mean() * 100

# Consistency: check invoice-level dates are not contradictory
invoice_dates = df.groupby("InvoiceNo")["InvoiceDate"].nunique()
consistency_score = (invoice_dates.le(1).mean() * 100)

# Uniqueness: exact duplicate rows
uniqueness_score = (1 - df.duplicated().mean()) * 100

# Validity: basic data types and business rules
valid_dates = df["InvoiceDate"].notna()
valid_quantity = df["Quantity"].gt(0)
valid_price = df["UnitPrice"].ge(0)

validity_score = (
    valid_dates & valid_quantity & valid_price
).mean() * 100

# Completeness across all cells
completeness_score = (1 - df.isna().sum().sum() / df.size) * 100

# Timeliness cannot be scored without a defined freshness SLA
quality_report = pd.DataFrame([
    ["Completeness", round(completeness_score, 2),
     "Measured missing cells"],
    ["Accuracy", round(accuracy_score, 2),
     "Proxy: positive quantity and non-negative price"],
    ["Consistency", round(consistency_score, 2),
     "Proxy: one distinct invoice date per invoice"],
    ["Timeliness", np.nan,
     "Requires ingestion timestamp and freshness SLA"],
    ["Uniqueness", round(uniqueness_score, 2),
     "No exact duplicate rows"],
    ["Validity", round(validity_score, 2),
     "Valid dates and basic numeric rules"]
], columns=["Dimension", "Score (%)", "Assessment"])

display(quality_report)
quality_report.to_csv("dama_quality_assessment.csv", index=False)

,Dimension,Score (%),Assessment
0,Completeness,96.82,Measured missing cells
1,Accuracy,98.03,Proxy: positive quantity and non-negative price
2,Consistency,99.83,Proxy: one distinct invoice date per invoice
3,Timeliness,NaN,Requires ingestion timestamp and freshness SLA
4,Uniqueness,100.00,No exact duplicate rows
5,Validity,98.03,Valid dates and basic numeric rules


In [10]:
import great_expectations as gx

print("Great Expectations version:", gx.__version__)

context = gx.get_context()
print("Data Context created:", type(context).__name__)

Great Expectations version: 1.24.1
Data Context created: EphemeralDataContext


In [11]:
# Create and register the suite with the Data Context
suite = gx.ExpectationSuite(
    name="online_retail_quality_suite"
)

suite = context.suites.add(suite)

# Add the eight expectations defined in your previous cell
for expectation in expectations:
    suite.add_expectation(expectation)

print("Expectations defined:", len(suite.expectations))
print("Suite registered successfully!")

Expectations defined: 8
Suite registered successfully!


In [13]:

# Create the Data Context
context = gx.get_context()

# Create a pandas datasource
datasource = context.data_sources.add_pandas(
    name="online_retail_datasource"
)

# Register the DataFrame as an asset
asset = datasource.add_dataframe_asset(
    name="online_retail_asset"
)

# Create a batch definition
batch_definition = asset.add_batch_definition_whole_dataframe(
    name="online_retail_batch"
)

# Attach your dataset
batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df}
)

# Run Great Expectations validation
validation_results = batch.validate(suite)

print(validation_results)

Calculating Metrics:   0%|          | 0/47 [00:00<?, ?it/s]

{
  "success": false,
  "results": [
    {
      "success": true,
      "expectation_config": {
        "type": "expect_column_values_to_not_be_null",
        "kwargs": {
          "batch_id": "online_retail_datasource-online_retail_asset",
          "column": "InvoiceNo"
        },
        "meta": {},
        "id": "6c4bd19e-5e86-4d7c-b17d-68a48a4bbf54",
        "severity": "critical"
      },
      "result": {
        "element_count": 536641,
        "unexpected_count": 0,
        "unexpected_percent": 0.0,
        "partial_unexpected_list": [],
        "partial_unexpected_counts": [],
        "partial_unexpected_index_list": []
      },
      "meta": {},
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": false,
      "expectation_config": {
        "type": "expect_column_values_to_be_unique",
        "kwargs": {
          "batch_id": "online_retail_datasource-online

In [15]:
# Create a fresh expectation suite
suite = gx.ExpectationSuite(name="online_retail_quality_suite_v2")

# 1. Invoice number must not be missing
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="InvoiceNo")
)

# 2. Stock code must not be missing
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="StockCode")
)

# 3. Quantity must not be missing
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="Quantity")
)

# 4. Quantity must be a positive value
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="Quantity", min_value=1
    )
)

# 5. Unit price must not be negative
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="UnitPrice", min_value=0
    )
)

# 6. Invoice date must not be missing
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="InvoiceDate")
)

# 7. Country must not be missing
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="Country")
)

# 8. Customer ID must not be missing
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="CustomerID")
)

# Register the new suite
suite = context.suites.add(suite)

print("Expectation suite created successfully.")
print("Total expectations:", len(suite.expectations))

Expectation suite created successfully.
Total expectations: 8


In [16]:
validation_results = batch.validate(suite)

print("Validation successful:", validation_results.success)
print("Total expectations:", len(validation_results.results))
print(
    "Passed:",
    sum(result.success for result in validation_results.results)
)
print(
    "Failed:",
    sum(not result.success for result in validation_results.results)
)

Calculating Metrics:   0%|          | 0/45 [00:00<?, ?it/s]

Validation successful: False
Total expectations: 8
Passed: 5
Failed: 3


In [17]:
results_table = pd.DataFrame([
    {
        "Expectation": result.expectation_config.type,
        "Column": result.expectation_config.kwargs.get("column"),
        "Passed": result.success,
        "Unexpected Count": result.result.get("unexpected_count", 0)
    }
    for result in validation_results.results
])

print(results_table.to_string(index=False))

                        Expectation      Column  Passed  Unexpected Count
expect_column_values_to_not_be_null   InvoiceNo    True                 0
expect_column_values_to_not_be_null   StockCode    True                 0
expect_column_values_to_not_be_null    Quantity    True                 0
 expect_column_values_to_be_between    Quantity   False             10587
 expect_column_values_to_be_between   UnitPrice   False                 2
expect_column_values_to_not_be_null InvoiceDate    True                 0
expect_column_values_to_not_be_null     Country    True                 0
expect_column_values_to_not_be_null  CustomerID   False            135037


In [18]:
print("Overall success:", validation_results.success)

for result in validation_results.results:
    status = "PASS" if result.success else "FAIL"
    print(
        status,
        "|",
        result.expectation_config.type,
        "| Column:",
        result.expectation_config.kwargs.get("column")
    )

Overall success: False
PASS | expect_column_values_to_not_be_null | Column: InvoiceNo
PASS | expect_column_values_to_not_be_null | Column: StockCode
PASS | expect_column_values_to_not_be_null | Column: Quantity
FAIL | expect_column_values_to_be_between | Column: Quantity
FAIL | expect_column_values_to_be_between | Column: UnitPrice
PASS | expect_column_values_to_not_be_null | Column: InvoiceDate
PASS | expect_column_values_to_not_be_null | Column: Country
FAIL | expect_column_values_to_not_be_null | Column: CustomerID


In [19]:
corrupted_df = df.copy()

# Error 1: Missing invoice number
corrupted_df.loc[corrupted_df.index[0], "InvoiceNo"] = None

# Error 2: Negative unit price
corrupted_df.loc[corrupted_df.index[1], "UnitPrice"] = -50

# Error 3: Missing invoice date
corrupted_df.loc[corrupted_df.index[2], "InvoiceDate"] = pd.NaT

print("Intentional errors introduced: 3")
print("Missing InvoiceNo:", corrupted_df["InvoiceNo"].isna().sum())
print("Negative UnitPrice:", (corrupted_df["UnitPrice"] < 0).sum())
print("Missing InvoiceDate:", corrupted_df["InvoiceDate"].isna().sum())

Intentional errors introduced: 3
Missing InvoiceNo: 1
Negative UnitPrice: 3
Missing InvoiceDate: 1


In [20]:
corrupted_batch = batch_definition.get_batch(
    batch_parameters={"dataframe": corrupted_df}
)

corrupted_results = corrupted_batch.validate(suite)

print("Corrupted dataset passes all checks:", corrupted_results.success)

for result in corrupted_results.results:
    status = "PASS" if result.success else "FAIL"
    print(
        status,
        "|",
        result.expectation_config.type,
        "| Column:",
        result.expectation_config.kwargs.get("column")
    )

Calculating Metrics:   0%|          | 0/45 [00:00<?, ?it/s]

Corrupted dataset passes all checks: False
FAIL | expect_column_values_to_not_be_null | Column: InvoiceNo
PASS | expect_column_values_to_not_be_null | Column: StockCode
PASS | expect_column_values_to_not_be_null | Column: Quantity
FAIL | expect_column_values_to_be_between | Column: Quantity
FAIL | expect_column_values_to_be_between | Column: UnitPrice
FAIL | expect_column_values_to_not_be_null | Column: InvoiceDate
PASS | expect_column_values_to_not_be_null | Column: Country
FAIL | expect_column_values_to_not_be_null | Column: CustomerID


In [21]:
# Create a summary of all validation results
error_test_results = pd.DataFrame([
    {
        "Expectation": result.expectation_config.type,
        "Column": result.expectation_config.kwargs.get("column"),
        "Passed": result.success,
        "Unexpected Count": result.result.get("unexpected_count", 0)
    }
    for result in corrupted_results.results
])

print(error_test_results.to_string(index=False))

                        Expectation      Column  Passed  Unexpected Count
expect_column_values_to_not_be_null   InvoiceNo   False                 1
expect_column_values_to_not_be_null   StockCode    True                 0
expect_column_values_to_not_be_null    Quantity    True                 0
 expect_column_values_to_be_between    Quantity   False             10587
 expect_column_values_to_be_between   UnitPrice   False                 3
expect_column_values_to_not_be_null InvoiceDate   False                 1
expect_column_values_to_not_be_null     Country    True                 0
expect_column_values_to_not_be_null  CustomerID   False            135037


In [22]:
# Save validation results
error_test_results.to_csv(
    "error_injection_test_results.csv",
    index=False
)

# Save the corrupted copy for evidence
corrupted_df.to_csv(
    "online_retail_corrupted_test.csv",
    index=False
)

print("Files saved successfully:")
print("1. error_injection_test_results.csv")
print("2. online_retail_corrupted_test.csv")

Files saved successfully:
1. error_injection_test_results.csv
2. online_retail_corrupted_test.csv


In [23]:
data_contract = {
    "dataContractSpecification": "0.9.3",
    "id": "online-retail-v1",
    "info": {
        "title": "Online Retail Data Contract",
        "version": "1.0.0",
        "status": "draft",
        "description": "Data quality rules for the Online Retail dataset used by a BI dashboard."
    },
    "roles": [
        {
            "role": "producer",
            "name": "Data Analyst"
        },
        {
            "role": "consumer",
            "name": "BI Dashboard"
        }
    ],
    "servers": [
        {
            "server": "local",
            "type": "filesystem",
            "path": "./"
        }
    ],
    "models": {
        "online_retail": {
            "description": "Retail transaction-level records.",
            "type": "table",
            "fields": {
                "InvoiceNo": {
                    "type": "string",
                    "required": True,
                    "description": "Invoice identifier."
                },
                "StockCode": {
                    "type": "string",
                    "required": True,
                    "description": "Product identifier."
                },
                "Description": {
                    "type": ["string", "null"],
                    "required": False,
                    "description": "Product description."
                },
                "Quantity": {
                    "type": "integer",
                    "required": True,
                    "description": "Quantity sold or returned."
                },
                "InvoiceDate": {
                    "type": "datetime",
                    "required": True,
                    "description": "Transaction date and time."
                },
                "UnitPrice": {
                    "type": "number",
                    "required": True,
                    "description": "Unit price; negative values require investigation."
                },
                "CustomerID": {
                    "type": ["string", "null"],
                    "required": False,
                    "description": "Customer identifier; treat as confidential."
                },
                "Country": {
                    "type": "string",
                    "required": True,
                    "description": "Customer country."
                }
            }
        }
    }
}

with open("data-contract-online-retail.yaml", "w", encoding="utf-8") as file:
    yaml.safe_dump(data_contract, file, sort_keys=False, allow_unicode=True)

print("Draft YAML data contract saved.")

Draft YAML data contract saved.


In [24]:
with open("data-contract-online-retail.yaml", "r", encoding="utf-8") as file:
    loaded_contract = yaml.safe_load(file)

print("YAML syntax valid:", isinstance(loaded_contract, dict))
print("Contract title:", loaded_contract["info"]["title"])
print("Fields defined:", len(loaded_contract["models"]["online_retail"]["fields"]))

YAML syntax valid: True
Contract title: Online Retail Data Contract
Fields defined: 8


In [25]:
pii_report = pd.DataFrame([
    {
        "Column": "InvoiceNo",
        "Classification": "Internal business data",
        "PII": "Not directly identifying",
        "Risk": "May reveal transaction activity",
        "Protection": "Restrict access where needed"
    },
    {
        "Column": "StockCode",
        "Classification": "Internal business data",
        "PII": "No obvious direct PII",
        "Risk": "May reveal product information",
        "Protection": "Role-based access"
    },
    {
        "Column": "Description",
        "Classification": "Review required",
        "PII": "Potentially, if free text contains identifiers",
        "Risk": "Unexpected personal details in text",
        "Protection": "Review and redact identifiers if found"
    },
    {
        "Column": "Quantity",
        "Classification": "Internal business data",
        "PII": "No obvious direct PII",
        "Risk": "Low personal-data risk",
        "Protection": "Standard access controls"
    },
    {
        "Column": "InvoiceDate",
        "Classification": "Internal business data",
        "PII": "Potentially identifying when linked to other data",
        "Risk": "Transaction timing patterns",
        "Protection": "Limit access or aggregate dates for reports"
    },
    {
        "Column": "UnitPrice",
        "Classification": "Internal business data",
        "PII": "No obvious direct PII",
        "Risk": "Commercial sensitivity",
        "Protection": "Restrict access where appropriate"
    },
    {
        "Column": "CustomerID",
        "Classification": "Confidential customer identifier",
        "PII": "Potentially identifying when linked to other data",
        "Risk": "Customer activity may be linked across transactions",
        "Protection": "Tokenisation or pseudonymisation; restrict access"
    },
    {
        "Column": "Country",
        "Classification": "General business data",
        "PII": "Not directly identifying by itself",
        "Risk": "May contribute to identification when combined with other data",
        "Protection": "Aggregate where appropriate"
    }
])

print(pii_report.to_string(index=False))
pii_report.to_csv("pii_classification_report.csv", index=False)

print("\nPII classification report saved.")

     Column                   Classification                                               PII                                                           Risk                                        Protection
  InvoiceNo           Internal business data                          Not directly identifying                                May reveal transaction activity                      Restrict access where needed
  StockCode           Internal business data                             No obvious direct PII                                 May reveal product information                                 Role-based access
Description                  Review required    Potentially, if free text contains identifiers                            Unexpected personal details in text            Review and redact identifiers if found
   Quantity           Internal business data                             No obvious direct PII                                         Low personal-data risk           

In [26]:
import pandas as pd

# 1. Completeness: percentage of non-missing cells
completeness_score = round(
    df.notna().to_numpy().mean() * 100, 2
)

# 2. Uniqueness: percentage of rows that are not exact duplicates
uniqueness_score = round(
    (1 - df.duplicated().mean()) * 100, 2
)

# 3. Validity: proportion passing selected business-rule checks
validity_checks = pd.DataFrame({
    "quantity_valid": df["Quantity"] >= 1,
    "unit_price_valid": df["UnitPrice"] >= 0,
    "invoice_date_valid": pd.to_datetime(
        df["InvoiceDate"], errors="coerce"
    ).notna()
})

validity_score = round(
    validity_checks.to_numpy().mean() * 100, 2
)

# 4. Consistency: simple formatting and type checks
country_consistent = (
    df["Country"].dropna().astype(str)
    .eq(df["Country"].dropna().astype(str).str.strip())
).mean() * 100

date_consistent = (
    pd.to_datetime(df["InvoiceDate"], errors="coerce").notna().mean()
    * 100
)

consistency_score = round(
    (country_consistent + date_consistent) / 2, 2
)

# 5. Accuracy: proxy based on price validity, not verified truth
accuracy_proxy = round(
    (df["UnitPrice"] >= 0).mean() * 100, 2
)

# 6. Timeliness: requires a freshness requirement and reference timestamp
timeliness_score = None

dama_report = pd.DataFrame([
    {
        "Dimension": "Completeness",
        "Score_Percent": completeness_score,
        "Assessment": "Percentage of non-missing cells",
        "Limitation": "Does not distinguish optional from required fields"
    },
    {
        "Dimension": "Uniqueness",
        "Score_Percent": uniqueness_score,
        "Assessment": "Rows without exact duplicates",
        "Limitation": "Repeated invoice numbers are allowed"
    },
    {
        "Dimension": "Validity",
        "Score_Percent": validity_score,
        "Assessment": "Selected quantity, price and date rules",
        "Limitation": "Negative quantities may represent returns"
    },
    {
        "Dimension": "Consistency",
        "Score_Percent": consistency_score,
        "Assessment": "Country formatting and date parseability",
        "Limitation": "Only selected consistency checks are measured"
    },
    {
        "Dimension": "Accuracy",
        "Score_Percent": accuracy_proxy,
        "Assessment": "Unit-price validity proxy only",
        "Limitation": "True accuracy requires a trusted reference"
    },
    {
        "Dimension": "Timeliness",
        "Score_Percent": timeliness_score,
        "Assessment": "Not scored",
        "Limitation": "No agreed freshness SLA or ingestion timestamp"
    }
])

dama_report.to_csv("dama_quality_assessment.csv", index=False)
print(dama_report.to_string(index=False))
print("\nDAMA quality report saved.")

   Dimension  Score_Percent                               Assessment                                         Limitation
Completeness          96.82          Percentage of non-missing cells Does not distinguish optional from required fields
  Uniqueness         100.00            Rows without exact duplicates               Repeated invoice numbers are allowed
    Validity          99.34  Selected quantity, price and date rules          Negative quantities may represent returns
 Consistency         100.00 Country formatting and date parseability      Only selected consistency checks are measured
    Accuracy         100.00           Unit-price validity proxy only         True accuracy requires a trusted reference
  Timeliness            NaN                               Not scored     No agreed freshness SLA or ingestion timestamp

DAMA quality report saved.


In [27]:
original_results_table = pd.DataFrame([
    {
        "Expectation": result.expectation_config.type,
        "Column": result.expectation_config.kwargs.get("column"),
        "Passed": result.success,
        "Unexpected_Count": result.result.get("unexpected_count", 0)
    }
    for result in validation_results.results
])

original_results_table.to_csv(
    "great_expectations_validation_results.csv",
    index=False
)

print(original_results_table.to_string(index=False))
print("\nOriginal validation results saved.")

                        Expectation      Column  Passed  Unexpected_Count
expect_column_values_to_not_be_null   InvoiceNo    True                 0
expect_column_values_to_not_be_null   StockCode    True                 0
expect_column_values_to_not_be_null    Quantity    True                 0
 expect_column_values_to_be_between    Quantity   False             10587
 expect_column_values_to_be_between   UnitPrice   False                 2
expect_column_values_to_not_be_null InvoiceDate    True                 0
expect_column_values_to_not_be_null     Country    True                 0
expect_column_values_to_not_be_null  CustomerID   False            135037

Original validation results saved.


In [28]:
# Show failures from the corrupted dataset
corrupted_results_table = pd.DataFrame([
    {
        "Expectation": result.expectation_config.type,
        "Column": result.expectation_config.kwargs.get("column"),
        "Passed": result.success,
        "Unexpected_Count": result.result.get("unexpected_count", 0)
    }
    for result in corrupted_results.results
])

print(corrupted_results_table.to_string(index=False))

# Check the three specific rules
target_columns = ["InvoiceNo", "UnitPrice", "InvoiceDate"]

for column in target_columns:
    matching = corrupted_results_table[
        corrupted_results_table["Column"] == column
    ]
    detected = not matching.empty and (~matching["Passed"]).any()
    print(f"{column} injected error detected: {detected}")

corrupted_results_table.to_csv(
    "error_injection_test_results.csv",
    index=False
)

                        Expectation      Column  Passed  Unexpected_Count
expect_column_values_to_not_be_null   InvoiceNo   False                 1
expect_column_values_to_not_be_null   StockCode    True                 0
expect_column_values_to_not_be_null    Quantity    True                 0
 expect_column_values_to_be_between    Quantity   False             10587
 expect_column_values_to_be_between   UnitPrice   False                 3
expect_column_values_to_not_be_null InvoiceDate   False                 1
expect_column_values_to_not_be_null     Country    True                 0
expect_column_values_to_not_be_null  CustomerID   False            135037
InvoiceNo injected error detected: True
UnitPrice injected error detected: True
InvoiceDate injected error detected: True


In [29]:
from pathlib import Path
import yaml

contract_path = Path("data-contract-online-retail.yaml")

with contract_path.open("r", encoding="utf-8") as file:
    loaded_contract = yaml.safe_load(file)

expected_columns = set(df.columns)
contract_columns = set(
    loaded_contract["models"]["online_retail"]["fields"].keys()
)

print("YAML parsed successfully:", isinstance(loaded_contract, dict))
print("Contract title:", loaded_contract["info"]["title"])
print("Dataset columns:", len(expected_columns))
print("Contract columns:", len(contract_columns))
print("Columns match:", expected_columns == contract_columns)

if expected_columns != contract_columns:
    print("Missing from contract:", expected_columns - contract_columns)
    print("Extra in contract:", contract_columns - expected_columns)

YAML parsed successfully: True
Contract title: Online Retail Data Contract
Dataset columns: 8
Contract columns: 8
Columns match: True


In [30]:
# Save the corrupted copy and its validation results
corrupted_df.to_csv(
    "online_retail_corrupted_test.csv", index=False
)

corrupted_results_table.to_csv(
    "error_injection_test_results.csv", index=False
)

# Confirm expected deliverables
required_files = [
    "dama_quality_assessment.csv",
    "great_expectations_validation_results.csv",
    "error_injection_test_results.csv",
    "online_retail_corrupted_test.csv",
    "data-contract-online-retail.yaml",
    "pii_classification_report.csv"
]

print("DELIVERABLE CHECKLIST")
for filename in required_files:
    file_path = Path(filename)
    print(
        "[OK]" if file_path.exists() else "[MISSING]",
        filename,
        f"({file_path.stat().st_size:,} bytes)" if file_path.exists() else ""
    )

DELIVERABLE CHECKLIST
[OK] dama_quality_assessment.csv (602 bytes)
[OK] great_expectations_validation_results.csv (485 bytes)
[OK] error_injection_test_results.csv (487 bytes)
[OK] online_retail_corrupted_test.csv (47,980,665 bytes)
[OK] data-contract-online-retail.yaml (1,462 bytes)
[OK] pii_classification_report.csv (1,149 bytes)


In [31]:
# Check whether the three intentional errors were detected

for column in ["InvoiceNo", "UnitPrice", "InvoiceDate"]:
    matching = corrupted_results_table[
        corrupted_results_table["Column"] == column
    ]

    if not matching.empty and (~matching["Passed"]).any():
        print(f"PASS: {column} error was detected.")
    else:
        print(f"CHECK: {column} error was not confirmed.")

PASS: InvoiceNo error was detected.
PASS: UnitPrice error was detected.
PASS: InvoiceDate error was detected.
